In [1]:
import os 
import torch 
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from dataset import DroneFaceDataset, get_embedding
from model import Embeddinghead, ArcFaceLoss


c:\Users\Shamm\anaconda33\envs\droneface2\lib\site-packages\mtcnn\mtcnn.py:34: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
import os
import shutil
import random
from PIL import Image
from deepface import DeepFace 
import numpy as np
from torch.utils.data import Dataset 
import torchvision.transforms as transforms

sources = ["open_data_set\photos_all_faces",
           "open_data_set\portraits",
           "open_data_set\\trio_gp",
           "open_data_set\\trio_cam"]

destination = "combined_dataset"
os.makedirs(destination, exist_ok=True)

for source in sources:
    for filename in os.listdir(source):
        if filename.lower().endswith((".jpg", ".png", ".jpeg")):
            src_path = os.path.join(source, filename)
            dst_path = os.path.join(destination, filename)
            shutil.copy(src_path, dst_path)

print("all images combined successfully")



source = "combined_dataset"
destination = "prepared"

os.makedirs(destination, exist_ok=True)

for filename in os.listdir(source):
    if filename.lower().endswith((".jpg", ".png", "jpeg")):
        identity = filename[0].upper()
        identity_folder = os.path.join(destination, identity)
        os.makedirs(identity_folder, exist_ok=True)
        src_path = os.path.join(source, filename)
        dst_path = os.path.join(identity_folder, filename)
        shutil.move(src_path, dst_path)

print("Data split by identity")


source = "prepared"
base_split = "split" 

#splitting ids into training, validation, and testing
train_ids = ["A", "B", "C", "D", "E", "F", "G", "H"] #70% Training
val_ids = ["I"] #10% validation
test_ids = ["J", "K"] #20% testing


for split, ids in [("train", train_ids), ("validation", val_ids), ("test", test_ids)]:
    for identity in ids: 
        src = os.path.join(source, identity)
        dst = os.path.join(base_split, split, identity)
        os.makedirs(dst, exist_ok=True)
        for file in os.listdir(src):
            shutil.copy(os.path.join(src, file),
                        os.path.join(dst, file))
            
print("identity disjoint split completed")


all images combined successfully
Data split by identity
identity disjoint split completed


In [3]:
#settings 
DEEPFACE_MODEL = "DeepFace"
Detector = "retineface"
Embedding_dim = 512
head_dim = 256
batch_Size = 32 
epochs = 30
lr = 1e-3 #learning rate 
threshold = 0.45 #above threshold:more confident, below  threshold=less confident

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using: {device}")


Using: cpu


In [4]:
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

#dataset loading
train_set = DroneFaceDataset("split\\train", augment=True)
validation_set = DroneFaceDataset("split\\validation", augment=False)
test_set = DroneFaceDataset("split\\test", augment=False)

num_classes = len(train_set.classes)
idx_to_class= {idx:name for name, idx in train_set.class_to_idx.items()}



**Extracting Embeddings**

In [5]:
if os.path.exists("cache_train_embeddings.npy"):
    print("loading train embeddings")
    train_embeddings = np.load("cache_train_embeddings.npy")
    train_labels = np.load("cache_train_labels.npy")
else: 
    print("extracting")
    train_embeddings, train_labels = [], []

    for identity in train_set.classes:
        folder= os.path.join("split", "train", identity)
        for filename in os.listdir(folder):
            if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            embedding = get_embedding(os.path.join(folder, filename))
            if embedding is not None:
                train_embeddings.append(embedding)
                train_labels.append(train_set.class_to_idx[identity])
            else:
                print("FAILED:", filename)
        print("processing:", filename)
    train_embeddings = np.array(train_embeddings)
    train_labels = np.array(train_labels)
    np.save("cache_train_embeddings.npy", train_embeddings)
    np.save("cache_train_labels.npy", train_labels)

if os.path.exists("cache_validation_embeddings.npy"):
    print("loading validation embeddings")
    validation_embeddings = np.load("cache_validation_embeddings.npy")
    validation_labels = np.load("cache_validation_labels.npy")
else: 
    print("extracting")
    validation_embeddings, validation_labels = [], []

    for identity in validation_set.classes:
        folder= os.path.join("split", "validation", identity)
        for filename in os.listdir(folder):
            if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            embedding = get_embedding(os.path.join(folder, filename))
            if embedding is not None:
                validation_embeddings.append(embedding)
                validation_labels.append(validation_set.class_to_idx[identity])

validation_embeddings = np.array(validation_embeddings)
validation_labels = np.array(validation_labels)
np.save("cache_validation_embeddings.npy", validation_embeddings)
np.save("cache_validation_labels.npy", validation_labels)

train_embeddings_tensor = torch.tensor(train_embeddings, dtype=torch.float32)
train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)
validation_embeddings_tensor = torch.tensor(validation_embeddings, dtype=torch.float32)
validation_labels_tensor = torch.tensor(validation_labels, dtype=torch.long)


extracting
processing: a_na_na_por_na.jpg
processing: b_gp_5_ef_30.jpg
processing: c_gp_5_ef_30.jpg
processing: d_na_na_por_na.jpg
processing: e_na_na_por_na.jpg
processing: f_gp_5_ef_30.jpg
processing: g_na_na_por_na.jpg
processing: h_gp_5_ef_30.jpg
extracting


In [6]:
print("Train embeddings shape:", train_embeddings_tensor.shape)
print("Train labels shape:", train_labels_tensor.shape)
print("Validation embeddings shape:", validation_embeddings_tensor.shape)
print("Validation labels shape:", validation_labels_tensor.shape)

Train embeddings shape: torch.Size([1044, 512])
Train labels shape: torch.Size([1044])
Validation embeddings shape: torch.Size([131, 512])
Validation labels shape: torch.Size([131])


**Model Set up**

In [7]:
head = Embeddinghead().to(device)
arcface = ArcFaceLoss(in_features=head_dim, num_classes=num_classes).to(device)

optimizer = torch.optim.Adam([{'params': filter(lambda p: p.requires_grad, head.parameters()), 'lr': lr},
                              {'params': arcface.parameters(), 'lr': lr}])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)



**Training Set up**

In [8]:
embedding_loader = DataLoader(TensorDataset(train_embeddings_tensor, train_labels_tensor),
                              batch_size=batch_Size,
                              shuffle=True)


best_valacc = 0.0

for epoch in range(30):
    head.train()
    arcface.train()
    total_loss = 0.0

    for embeddings, labels in embedding_loader:
        embeddings=embeddings.to(device)
        labels=labels.to(device)
        optimizer.zero_grad()
        proj = head(embeddings) #compresses from 4096 to 256
        logits = arcface(proj, labels)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss +=loss.item()
    scheduler.step()
    #building gallery
    head.eval()
    embeddings_gallery, labels_gallery = [], []
    with torch.no_grad():
        for images, labels in DataLoader(TensorDataset(train_embeddings_tensor, train_labels_tensor), batch_size=batch_Size, shuffle=False):
            embeddings_gallery.append(head(embeddings.to(device)).cpu())
            labels_gallery.append(labels)
    embeddings_gallery = torch.cat(embeddings_gallery)
    labels_gallery = torch.cat(labels_gallery)

    #validation accuracy using cosine similarity (Different people)
    val_proj = []
    with torch.no_grad():
        for embeddings, _ in DataLoader(TensorDataset(validation_embeddings_tensor, validation_labels_tensor), batch_size=batch_Size, shuffle=False):
            val_proj.append(head(embeddings.to(device)).cpu())
        val_proj = torch.cat(val_proj)
        correct= 0
        for i in range(len(embeddings)):
            similarities = F.cosine_similarity(val_proj[i].unsqueeze(0), embeddings_gallery)
            best_match = torch.argmax(similarities).item()
            if labels_gallery[best_match].item() == validation_labels_tensor[i].item():
                correct +=1
        validation_accuracy = 100 * correct / len(val_proj)
                               
    
    #to save best model 
    if validation_accuracy > best_valacc:
        best_valacc = validation_accuracy
        torch.save({'head': head.state_dict(), 'arcface': arcface.state_dict(), "epoch": epoch + 1}, "best_model.pth")
        print("best model saved")
        saved = "Saved"
    else: 
        saved = ""
    print(f"Epoch: {epoch+1}    Loss: {total_loss:.4f}    Validation Accuracy: {validation_accuracy:.2f}%")
    print(f" Best Validation Accuracy: {best_valacc:.2f}%")

   
    
print(f"Best validation accuracy: {best_valacc:.2f}%") 

best model saved
Epoch: 1    Loss: 440.5075    Validation Accuracy: 2.29%
 Best Validation Accuracy: 2.29%
Epoch: 2    Loss: 241.0493    Validation Accuracy: 2.29%
 Best Validation Accuracy: 2.29%
Epoch: 3    Loss: 224.4037    Validation Accuracy: 2.29%
 Best Validation Accuracy: 2.29%
Epoch: 4    Loss: 185.2210    Validation Accuracy: 2.29%
 Best Validation Accuracy: 2.29%
Epoch: 5    Loss: 180.3857    Validation Accuracy: 2.29%
 Best Validation Accuracy: 2.29%
Epoch: 6    Loss: 127.0368    Validation Accuracy: 2.29%
 Best Validation Accuracy: 2.29%
Epoch: 7    Loss: 123.6307    Validation Accuracy: 2.29%
 Best Validation Accuracy: 2.29%
Epoch: 8    Loss: 120.8592    Validation Accuracy: 2.29%
 Best Validation Accuracy: 2.29%
Epoch: 9    Loss: 95.1913    Validation Accuracy: 2.29%
 Best Validation Accuracy: 2.29%
Epoch: 10    Loss: 84.4434    Validation Accuracy: 2.29%
 Best Validation Accuracy: 2.29%
Epoch: 11    Loss: 78.0723    Validation Accuracy: 2.29%
 Best Validation Accuracy: 

**Testing the model**

In [9]:
#loading the best saved model
checkpoint =torch.load("best_model.pth", map_location=device)
head.load_state_dict(checkpoint["head"])
head.eval()

top1 = top5 = total = 0

for identity in test_set.classes: 
    folder = os.path.join("split", "test", identity)
    true_label = test_set.class_to_idx[identity]
    for filename in os.listdir(folder):
        if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        embedding = get_embedding(os.path.join(folder, filename))
        if embedding is None: 
            continue
        embedding_tensor = torch.tensor(embedding, dtype = torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            proj = head(embedding_tensor).cpu().squeeze(0)

        similarities = F.cosine_similarity(proj.unsqueeze(0), embeddings_gallery)
        top5_idx = torch.topk(similarities, k=5).indices

        if labels_gallery[top5_idx[0]].item() == true_label:
            top1 +=1
        if true_label in [labels_gallery[i].item() for i in top5_idx]:
            top5 +=1
        total +=1

print(f"Rank 1 Accuracy: {100*top1/total:.2f}%")
print(f"Rank 5 Accuracy: {100*top5/total:.2f}%"
      )

Rank 1 Accuracy: 50.00%
Rank 5 Accuracy: 50.00%


In [10]:
#testing on a single image

def recognize_image(img_path):
    embedding = get_embedding(img_path)
    if embedding is None:
        print("No face found")
        return
    embedding_tensor = torch.tensor(embedding, dtype = torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        proj = head(embedding_tensor).cpu().squeeze(0)
    similarities = F.cosine_similarity(proj.unsqueeze(0), embeddings_gallery)
    top5_idx = torch.topk(similarities, k=5).indices

    best_score = similarities[top5_idx[0]].item()
    best_name = idx_to_class[labels_gallery[top5_idx[0]].item()]
    predicted = best_name if best_score >= threshold else "unknown"

    print(f"Predicted: {predicted} score: {best_score:.4f}")
    for i, idx in enumerate(top5_idx, 1):
        name = idx_to_class[labels_gallery[idx].item()]
        score = similarities[idx].item()
        print(f"{i} {name} {score:.4f}")